<a href="https://colab.research.google.com/github/Renan-RodriguesDEV/ChatBots/blob/main/whisper_with_openai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Instalação das blibliotecas (openai-whisper, gTTS, openai, python-dotenv)
!pip install openai-whisper gTTS openai python-dotenv -q

## Função para obter o audio de input

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
import base64

# Função JS para capturar o microfone do navegador
def record_audio():
    js = Javascript('''
        async function recordAudio() {
            const div = document.createElement('div');
            const button = document.createElement('button');
            button.textContent = 'Clique para Gravar';
            document.body.appendChild(div);
            div.appendChild(button);

            const stream = await navigator.mediaDevices.getUserMedia({audio: true});
            const recorder = new MediaRecorder(stream);
            const chunks = [];

            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.start();

            await new Promise(resolve => button.onclick = resolve);
            recorder.stop();

            return new Promise(resolve => {
                recorder.onstop = async () => {
                    const blob = new Blob(chunks);
                    const reader = new FileReader();
                    reader.readAsDataURL(blob);
                    reader.onloadend = () => resolve(reader.result);
                };
            });
        }
    ''')
    display(js)
    data = output.eval_js('recordAudio()')

    # Decodifica e salva o arquivo
    binary = base64.b64decode(data.split(',')[1])
    with open('audio.wav', 'wb') as f:
        f.write(binary)
    print("Áudio gravado com sucesso!")

# Para usar, basta chamar:
record_audio()

## Função para trancrever o audio em texto (spech-to-text)

In [ ]:
# Primeiro, instale a biblioteca: !pip install openai-whisper
import whisper

def transcribe_audio(file_path):
    # Carregamos o modelo "base" (rápido e eficiente)
    model = whisper.load_model("base")

    # O Whisper processa o áudio e nos devolve o texto
    result = model.transcribe(file_path)
    return result["text"]

texto_usuario = transcribe_audio('audio.wav')
print(f"Você disse: {texto_usuario}")

## Função para pergunta ao ChatGPT

In [ ]:
# Instale a biblioteca: !pip install openai
from openai import OpenAI

# Configure sua chave da API aqui
CHAVE_DA_API = os.getenv('OPENAPI_KEY')
client = OpenAI(api_key=CHAVE_DA_API)

def ask_chatgpt(question):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Você é um assistente útil e conciso."},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

resposta_ia = ask_chatgpt(texto_usuario)

## Função para converter texto em audio

In [ ]:
# Instale a biblioteca: !pip install gTTS
from gtts import gTTS
from IPython.display import Audio

def speak_response(text):
    # Converte o texto para áudio em português
    tts = gTTS(text=text, lang='pt')
    tts.save("response.mp3")

    # Exibe o player de áudio no Colab
    return Audio("response.mp3", autoplay=True)

speak_response(resposta_ia)